# Week 3 Problem Set
## The donor's \$20 million

**Goal of this problem set:** reproduce the +0.7 percentage point average treatment effect on the actual replication data, compute the ATE for a single state, run a randomization inference test to show the gap is bigger than chance, and examine where the effect is bigger and smaller. Then write a 250–350 word memo to the nonprofit board recommending whether to fund the donor's \$20M scale-up of the report card mailer for 2026.

**What you'll hand in:**
- This notebook, with all code cells run.
- A 250–350 word memo in the markdown cell at the end.

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the **Colab guide**, linked from the syllabus.

## Setup

Run the cell below to load the data.

In [ ]:
import pandas as pd, numpy as np
import statsmodels.formula.api as smf

df = pd.read_csv(
    'https://raw.githubusercontent.com/joshuakalla/'
    'data_science_campaigns_26/main/weeks/'
    'wk03_rcts_and_ate/data/gotv_2014.csv')
df.head()

## Task 1: Reproduce the headline ATE

The code below is the same `groupby` move from livecode. **Run it.** It prints the control mean, the treated mean, and the difference. The difference should be about +0.7 percentage points.

Then write **one sentence** in the markdown cell after it describing what the output means in plain English, and how it compares to the donor's quoted number from 2008 (+4.8 pp).

In [ ]:
turnout_by_arm = df.groupby('treatment')['voted_2014'].mean()
control_mean = turnout_by_arm.loc[0]
treated_mean = turnout_by_arm.loc[1]
observed_ate = treated_mean - control_mean

print('Control group turnout:  ', round(100*control_mean, 3), '%')
print('Treatment group turnout:', round(100*treated_mean, 3), '%')
print('Observed ATE:            ', round(100*observed_ate, 3),
      'percentage points')

**Your one sentence (replace this text):**

*…*

### The ATE as a regression, and what "controlling for" does here

You computed the ATE as a difference in group means. As in Week 2, it is also a **regression coefficient**. Run this:

In [ ]:
smf.ols('voted_2014 ~ treatment', data=df).fit().params

The `treatment` coefficient should match your ATE (about **0.0073 = +0.73 pp**). Because treatment was assigned by a coin flip, this coefficient is a *causal* effect, not just an association.

**Before you move on:** the cell below adds vote history as a **control**. In Week 2, adding a control cut the program coefficient by more than half. **Predict first:** will the `treatment` coefficient here change by a lot, or only a little, and why? Then run the cell and check.

In [ ]:
smf.ols('voted_2014 ~ treatment + C(vote_history_stratum)',
        data=df).fit().params['treatment']

**Your answer (2–3 sentences):** The coefficient moved from about 0.0073 to 0.0063, a real change (roughly 14%), but far smaller than Week 2's, where a control cut the coefficient by more than half. Why does controlling for vote history change the estimate *less* here than controlling for prior turnout did in Week 2? (Hint: what did the coin flip already do to the two groups?)

*Replace this text with your answer.*

## Task 2: ATE in one state

The dataset has a column called `state` with two-letter state codes. Pick one state (e.g., `'FL'` for Florida). In the two cells below:

1. **Filter the data.** Create a new DataFrame containing only voters from your chosen state. Use this filtering pattern: `florida = df[df['state'] == 'FL']`. Print `florida.shape` to confirm it's smaller than the full dataset.

2. **Compute the ATE.** Using your filtered DataFrame, compute the ATE with the same `groupby` pattern from Task 1. Print the control mean, treated mean, and difference.

*Self-check: if you pick `'FL'` you should get about **+1.2 pp** on roughly 8,000 voters. Different states give different numbers, and small states (a few hundred voters) give wild, noisy ones. Sampling variability!*

In [ ]:
# Step 1: filter to one state
one_state = # YOUR CODE HERE

In [ ]:
# Step 2: compute the ATE for your state
# YOUR CODE HERE
# End with a line assigning the difference to `state_ate`.

state_ate

**Your one sentence:** how does your state’s ATE compare to the overall +0.7 pp? Is it larger, smaller, or about the same?

*Replace this text with your answer.*

## Task 3: Randomization inference

Could a gap of +0.7 percentage points have happened just by chance, if the mailer did literally nothing? We answer this in code. The procedure (same as live coding):

1. Take the actual outcomes as fixed.
2. Re-randomize: assign treatment again at random, keeping the same number of voters in each group.
3. Recompute the difference of means under this fake assignment.
4. Do this 1,000 times.
5. Compare our actual observed ATE to the distribution of fake ATEs.

**Run the cell below.** It will produce a p-value: the fraction of fake assignments that produced a gap as extreme as the one we actually observed, in either direction.

*Your number will not match the one from class.* Live coding used a different starting seed and got 8 out of 1,000; this cell uses another one and gets about 14 out of 1,000. Both are correct. The procedure is random, so the count moves a little and the conclusion does not.

In [ ]:
# Same procedure as livecode, but with a different seed
# for fresh randomness.
np.random.seed(42)

n_reps = 1000
fake_ates = np.empty(n_reps)

for i in range(n_reps):
    # Shuffle the treatment assignments, the same
    # .sample(frac=1).values pattern as livecode. The outcomes
    # (voted_2014) never move. Only the assignment changes.
    shuffled_treatment = df['treatment'].sample(frac=1).values
    # Compute the fake ATE under this new assignment
    fake_treated = df['voted_2014'][shuffled_treatment == 1]
    fake_control = df['voted_2014'][shuffled_treatment == 0]
    fake_ates[i] = fake_treated.mean() - fake_control.mean()

fake_ates_pp = 100 * fake_ates
observed_ate_pp = 100 * observed_ate

# How many fake re-assignments produced a gap as extreme as
# our real one?
n_as_extreme = int(np.sum(
    np.abs(fake_ates_pp) >= abs(observed_ate_pp)))
p_value = n_as_extreme / n_reps

print(f'Observed ATE:                                    '
      f'{observed_ate_pp:+.3f} pp')
print(f'Number of fake assignments with |ATE| as large:  '
      f'{n_as_extreme} out of {n_reps}')
print(f'Two-sided randomization-inference p-value:       '
      f'{p_value:.4f}')

**Your one sentence on the p-value (replace this text):** *In plain English, what does this p-value tell us about whether the mailer really did something?*

*…*

**Before you move on:** In the loop above, `df['treatment']` (the assignment) gets shuffled on every iteration, but `df['voted_2014']` (the outcome) is never touched. Nobody's vote changes.

Why is leaving the outcomes alone what makes this a test of the null hypothesis at all? Put another way: what is the null hypothesis claiming about a voter's outcome that lets us keep her vote fixed while we move her into the other group?

Write your answer in 2–3 sentences in the cell below.

**Your answer:**

*Replace this text with your answer.*

## Task 4: Where the effect lives

The code below computes the ATE separately by `vote_history_stratum` (how often the voter has voted in past elections, relative to their state median) and by `high_salience_state` (the paper's own 1-to-4 score for how much was at stake in the state in 2014, counting a contested Senate race, a contested governor's race, and a Cook Political Report toss-up rating for each; states scoring 3 or 4 are coded 1). **Run both cells.**

Then write **two sentences**: one describing where the biggest effects show up in the vote-history table, and one describing what the salience table tells you about the disagreement with the 2008 study (which was a low-salience August primary).

In [ ]:
table_vh = (df
    .groupby(['vote_history_stratum', 'treatment'])['voted_2014']
    .mean()
    .unstack('treatment')
    .rename(columns={0: 'control', 1: 'treated'}))
table_vh['ATE_pp'] = 100 * (table_vh['treated']
                            - table_vh['control'])
table_vh.round(4)

**Before you move on:** If the mailer had exactly the same effect on every voter regardless of their vote history, what would the `ATE_pp` column look like? Would the three numbers be identical, or would they still differ a bit?

The three numbers in front of you *are* different. What would you need to know before concluding that the mailer really works differently on different kinds of voters?

Write your answer in 1–2 sentences in the cell below.

**Your answer:**

*Replace this text with your answer.*

In [ ]:
table_sal = (df
    .groupby(['high_salience_state', 'treatment'])['voted_2014']
    .mean()
    .unstack('treatment')
    .rename(columns={0: 'control', 1: 'treated'}))
table_sal['ATE_pp'] = 100 * (table_sal['treated']
                             - table_sal['control'])
table_sal.round(4)

**Your two sentences (replace this text):**

*…*

## Task 5: The memo

You are a research analyst at a campaign-funding nonprofit. The board meets in two weeks. A donor has asked the board to commit **\$20 million** to scale the Voting Report Card mailer across competitive 2026 House districts. Her case rests on the 2008 paper (+4.8 pp). You have just verified, on the actual 2014 replication data, that the same report card produces about **+0.7 pp** at scale in a midterm general election, and that this gap is unlikely to be chance (your randomization inference p-value should come out around 0.014).

**Cost-per-vote arithmetic, for reference.** The donor proposes \$20M at roughly \$1 per mailer = 20 million households mailed.
- **Under +4.8 pp:** 0.048 × 20,000,000 = 960,000 extra votes ⇒ \$20M / 960k ≈ **\$21 per vote**.
- **Under +0.7 pp:** 0.007 × 20,000,000 = 140,000 extra votes ⇒ \$20M / 140k ≈ **\$143 per vote**.

In the markdown cell below, write a **250–350 word memo** to the board.

### Three things your memo must do

1. **Defend a clear recommendation.** Yes, no, or "yes with the following conditions." Don't write "we need more research." That is always true and is not a recommendation.
2. **Reconcile the two studies.** Both numbers are real. Pick one as the better guess for what would happen in 2026 and give a *specific, concrete* reason grounded in the actual differences between the 2006 Michigan primary and the 2014 17-state midterm. ("Internal validity vs. external validity" is not an answer; it is the vocabulary you use to explain your answer. *Your* answer has to name a specific feature.)
3. **Steelman the donor under the 0.7-point number, not the 4.8-point number.** Construct the strongest version of "fund the rollout anyway, even at +0.7 percentage points and \$143 per vote" that you can think of, and either accept it (modify your recommendation accordingly) or specifically explain why it isn't enough.

### Two style rules

- **State your recommendation in the very first sentence.**
- **250–350 words.** The board will read 300. They will not read 600.

**Memo to:** Nonprofit Board
**From:** You, Research Analyst
**Re:** Donor proposal: \$20M Voting Report Card scale-up for 2026

*Replace this text with your 250–350 word memo.*

---

**Due at 4:00pm on Wednesday Sep 23**, to the **problem set** assignment on Canvas. Whatever you had at 6:00pm in class already went to the separate **in-class** assignment; that one is your attendance credit and you do not resubmit it.


## Before you submit

1. **Runtime → Restart session and run all.** Do this *after* you have finished every task and written your memo. It clears every variable and runs the notebook from top to bottom, in order, so the version you hand in is one that actually works start to finish.
2. **Check that every cell actually ran.** Scroll from the top. Every code cell should show a number in its left margin and its output below it. If the run stopped at a cell with an error, that is a cell you have not finished. Fix it, then restart and run all again.
3. **File → Print → Save as PDF.**
4. **Open the PDF and read it before you upload.** The PDF will look complete even when it isn't. Every heading and prompt prints whether or not the code ran. What matters is the **output**: under each code cell you should see a table, a number, or a plot. A red error box, or `[ ]` with nothing beneath it, means that part did not run and will be graded as missing. Also check that your memo printed in full and that no plot is cut off at a page break.
5. Upload the PDF to Canvas.